# realworld2.ipynb

This notebook was somewhat inspired by a Medium article I recently read which suggests using SQL instead of pandas for increased efficiency.

I will try to rewrite parts of realworld.ipynb to try out this concept.

I already figured out how to store numpy arrays in SQLite without converting to text, by using WKT for example. See `np2sqlite.py`.

In [1]:
from np2sqlite import array2blob, blob2array
from roadside import test_build_db, get_config
import numpy as np
from icecream import ic
import os
import sqlite3
import cv2

from pyefd import elliptic_fourier_descriptors, reconstruct_contour
from shapely.wkt import loads

# import pandas as pd

roadside


# Functions

In [2]:
def reconstruct_aligned_mask(image_shape, contour, order=10, align_to_centroid=True):
    """
    Finds EFDs and reconstructs the mask perfectly aligned with the original locus.
    
    Parameters:
        image_shape (tuple): Shape of the original image (H, W)
        contour (ndarray): Contour array of original image; shape (N, 2) or (N, 1, 2)
        order (int): Number of Fourier coefficients to use
        
    Returns:
        ndarray: Binary mask with the reconstructed shape in the correct position
    """
    
    ic()
    
    # 1. Standardize contour shape to (N, 2)
    contour = contour.reshape(-1, 2)
    
    # 2. Calculate the true centroid (locus) of the original contour using moments.
    # This keeps the reconstructed shape strictly bound to the true defect location.
    M = cv2.moments(contour)
    if M["m00"] != 0:
        cX = M["m10"] / M["m00"]
        cY = M["m01"] / M["m00"]
    else:
        cX, cY = np.mean(contour, axis=0)

    # 3. Compute EFD coefficients (keeping unnormalized to retain spatial properties)
    coeffs = elliptic_fourier_descriptors(contour, order=order, normalize=False)
    
    # 4. Corrected function: Reconstruct contour points via the native API.
    # We pass the calculated cX, cY into the locus argument.
    # Next line added by Aubrey Moore 2026-06-02
    num_points = contour.shape[0]  # Use the original number of contour points for reconstruction
    reconstructed_points = reconstruct_contour(coeffs, locus=(cX, cY), num_points=num_points)
    
    # 5. Prevent sub-pixel "floor bias" shift by rounding before converting to integer
    reconstructed_contour = np.round(reconstructed_points).astype(np.int32)
    reconstructed_contour = reconstructed_contour.reshape(-1, 1, 2)
    
    # 6. Create the aligned mask
    reconstructed_mask = np.zeros(image_shape, dtype=np.uint8)
    cv2.drawContours(reconstructed_mask, [reconstructed_contour], -1, 255, -1)
    
    if align_to_centroid:
        # Calculate the centroid of the reconstructed mask
        M_recon = cv2.moments(reconstructed_contour)
        if M_recon["m00"] != 0:
            recon_cX = M_recon["m10"] / M_recon["m00"]
            recon_cY = M_recon["m01"] / M_recon["m00"]
        else:
            recon_cX, recon_cY = np.mean(reconstructed_contour.reshape(-1, 2), axis=0)
        
        # Calculate the shift needed to align the reconstructed contour's centroid with the original
        shift_x = int(cX - recon_cX)
        shift_y = int(cY - recon_cY)
        ic(shift_x, shift_y)
        
        # Shift the reconstructed contour and mask
        translation_matrix = np.float32([[1, 0, shift_x], [0, 1, shift_y]])
        reconstructed_mask = cv2.warpAffine(reconstructed_mask, translation_matrix, (image_shape[1], image_shape[0]))
        reconstructed_contour = cv2.transform(reconstructed_contour, translation_matrix)
    
    return reconstructed_contour, reconstructed_mask


In [3]:
def get_centroid(contour):
    """ Returns centroid of a contour. """
    M = cv2.moments(contour)
    if M["m00"] != 0:
        cX = int(M["m10"] / M["m00"])
        cY = int(M["m01"] / M["m00"])
    else:
        cX, cY = np.mean(contour, axis=0)
    return cX, cY    

In [4]:
def calc_defect_contours(image_height, image_width, tree_wkt, order, minpixels):  
    poly = loads(tree_wkt)
    coords = list(poly.exterior.coords)    
    tree_contour = np.array(coords, dtype=np.int32).reshape(-1, 1, 2)
    
    canvas = np.zeros((image_height, image_width), np.uint8)
    tree_mask = cv2.drawContours(canvas, [tree_contour], -1, 255, -1)
    tree_mask_cx, tree_mask_cy = get_centroid(tree_mask)
    
    _, reconstructed_mask = reconstruct_aligned_mask(image_shape=(image_height, image_width), contour=tree_contour, order=order)
    # reconstructed_mask_cx, reconstructed_mask_cy = get_centroid(reconstructed_mask)
    registered_mask = reconstructed_mask.copy()
    additions_mask = cv2.bitwise_and(registered_mask, cv2.bitwise_not(tree_mask))    
    defect_contours, _ = cv2.findContours(additions_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    defect_contours = [cnt for cnt in defect_contours if cv2.contourArea(cnt) > minpixels]

    return defect_contours

# MAIN

In [20]:
config = get_config()
for key, value in config.items():
    ic(key, value)

ic| key: 'default_schema_sql'
    value: '''    CREATE TABLE IF NOT EXISTS images (
                    image_id INTEGER PRIMARY KEY AUTOINCREMENT,
                    image_path TEXT UNIQUE,
                    image_width INTEGER,
                    image_height INTEGER,
                    timestamp TEXT,
                    latitude REAL,
                    longitude REAL
                );
                CREATE TABLE IF NOT EXISTS detections (
                    detection_id INTEGER PRIMARY KEY AUTOINCREMENT,
                    image_id INTEGER,
                    class_id INTEGER,
                    tree_wkt TEXT,
                    crown_wkt TEXT,
                    x_min INTEGER,
                    y_min INTEGER,
                    x_max INTEGER,
                    y_max INTEGER,
                    confidence REAL,
                    FOREIGN KEY (image_id) REFERENCES images (image_id) ON DELETE CASCADE 
                );
                CREATE TABLE IF NOT EXISTS

In [6]:
if not os.path.exists(config['dbpath']):
    test_build_db()

conn = sqlite3.connect(config['dbpath'])
# conn.row_factory = sqlite3.Row # allows us to access columns by name

# FOR TESTING, WE WILL REMOVE ALL DATA FROM THE DAMAGE TABLE TO START FRESH
conn.execute('DELETE FROM damage')
conn.commit()

sql = 'select image_id, image_path, image_width, image_height from images'
for image_row in conn.execute(sql):
    image_id, image_path, image_width, image_height = image_row
    
    sql = 'SELECT detection_id, tree_wkt FROM detections WHERE image_id=?'
    for detection_row in conn.execute(sql, (image_id,)):
        detection_id, tree_wkt = detection_row
        ic('processing image', image_id, detection_id)
        
        defect_contours = calc_defect_contours(image_height, image_width, tree_wkt, config['order'], config['minpixels'])
        ic(len(defect_contours))       
        for cnt in defect_contours:
            cnt_blob = array2blob(cnt)
            conn.execute('INSERT INTO damage (detection_id, defect_contour) VALUES (?, ?)', (detection_id, cnt_blob))
conn.commit()

conn.close()

ic('FINISHED');

ic| 'processing image', image_id: 1, detection_id: 1
ic| 3337628227.py:14 in reconstruct_aligned_mask() at 10:35:38.573
ic| shift_x: 3, shift_y: -41
ic| len(defect_contours): 14
ic| 'processing image', image_id: 1, detection_id: 2
ic| 3337628227.py:14 in reconstruct_aligned_mask() at 10:35:38.603
ic| shift_x: 0, shift_y: 41
ic| len(defect_contours): 20
ic| 'processing image', image_id: 2, detection_id: 3
ic| 3337628227.py:14 in reconstruct_aligned_mask() at 10:35:38.626
ic| shift_x: 10, shift_y: 85
ic| len(defect_contours): 13
ic| 'processing image', image_id: 2, detection_id: 4
ic| 3337628227.py:14 in reconstruct_aligned_mask() at 10:35:38.645
ic| shift_x: 3, shift_y: 5
ic| len(defect_contours): 2
ic| 'processing image', image_id: 2, detection_id: 5
ic| 3337628227.py:14 in reconstruct_aligned_mask() at 10:35:38.662
ic| shift_x: -3, shift_y: 5
ic| len(defect_contours): 0
ic| 'processing image', image_id: 2, detection_id: 6
ic| 3337628227.py:14 in reconstruct_aligned_mask() at 10:35:38.